In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F



In [ ]:
stoi={s:i for i,s in enumerate(string.ascii_lowercase)}
stoi

{'a': 0,
 'b': 1,
 'c': 2,
 'd': 3,
 'e': 4,
 'f': 5,
 'g': 6,
 'h': 7,
 'i': 8,
 'j': 9,
 'k': 10,
 'l': 11,
 'm': 12,
 'n': 13,
 'o': 14,
 'p': 15,
 'q': 16,
 'r': 17,
 's': 18,
 't': 19,
 'u': 20,
 'v': 21,
 'w': 22,
 'x': 23,
 'y': 24,
 'z': 25}

In [12]:
example='love'
idx=[]

for i in example:
    idx.append(stoi[i])

idx=torch.tensor(idx)
idx

tensor([11, 14, 21,  4])

In [13]:
#使用独热编码,将文本转换为二维张量
num=len(stoi.keys())
x=F.one_hot(idx,num_classes=num).float()
x,x.shape

(tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 1., 0., 0., 0., 0.],
         [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0.]]),
 torch.Size([4, 26]))

In [14]:
dims=5
w=torch.randn(num,dims)
(x @ w),(x @ w).shape

(tensor([[-0.6761, -0.0552,  0.0930,  0.3280, -0.2793],
         [-1.8303, -0.1517,  1.2021, -0.7420,  0.3395],
         [ 0.1422,  1.9304,  0.1664, -1.0745,  0.3456],
         [ 1.1685, -1.7064,  1.6178, -0.2357,  1.2747]]),
 torch.Size([4, 5]))

In [16]:
w[idx],w[idx].shape

(tensor([[-0.6761, -0.0552,  0.0930,  0.3280, -0.2793],
         [-1.8303, -0.1517,  1.2021, -0.7420,  0.3395],
         [ 0.1422,  1.9304,  0.1664, -1.0745,  0.3456],
         [ 1.1685, -1.7064,  1.6178, -0.2357,  1.2747]]),
 torch.Size([4, 5]))

In [18]:
class Embedding:
    def __init__(self,num_embeddings,embedding_dims):
        self.weight=torch.randn(num_embeddings,embedding_dims,requires_grad=True)
    
    def __call__(self,x):
        self.out=self.weight[x]
        return self.out
    
    def parameters(self):
        return [self.weight]


In [19]:
em=Embedding(num,5)
x=torch.randint(0,num,(10, ))
em(x).shape

torch.Size([10, 5])

In [ ]:
x=torch.randint(0,num,(20,10)) # 20个文本  10 每一个文本的长度 5使用的嵌入维度，5个特征
em(x).shape

torch.Size([20, 10, 5])

In [21]:
torch.cuda.is_available()

True

In [7]:
words=open("names.txt","r").read().splitlines()
words[:3]

['emma', 'olivia', 'ava']

In [8]:
class CharTokenizer:
    def __init__(self,data,begin_ind=0,end_ind=1):
        chars=sorted(list(set(''.join(words))))
        self.stoi={s:i+2 for i,s in enumerate(chars)}
        self.stoi['<B>']=begin_ind
        self.stoi['<E>']=end_ind  
        self.itos={i:s for s,i in self.stoi.items()}
        self.begin_ind=begin_ind
        self.end_ind=end_ind
    
    def encode(self,x):
        return [self.stoi[i] for i in x]

    def decode(self,x):
        return [self.itos[i] for i in x]


In [13]:
tokenizer=CharTokenizer(words)
test_str='harrypotter'
re=tokenizer.encode(test_str)
re
''.join(tokenizer.decode(re))


'harrypotter'

In [14]:
def autoregressive_trans(text,tokenizer,context_size=3):
    # 将文本转换为索引
    idx=tokenizer.encode(text)
    # 添加开始和结束标记
    idx=[tokenizer.begin_ind]+idx+[tokenizer.end_ind]
    # 创建输入和输出序列
    x,y=[],[]
    for i in range(len(idx)-context_size):
        x.append(idx[i:i+context_size])
        y.append(idx[i+context_size])
    return torch.tensor(x),torch.tensor(y)

In [16]:
def process(data,tokenizer):
    inputs,labels=[],[]
    for w in words:
        i,l = autoregressive_trans(w,tokenizer)
        inputs.append(i)
        labels.append(l)
    return torch.cat(inputs,dim=0),torch.cat(labels,dim=0)


In [18]:
process(words,tokenizer)

NameError: name 'torch' is not defined